# Phase 3 — Création de la cible « canular »

Objectifs :
- Recharger les lignes valides du fichier.
- Refaire les conversions de types nécessaires.
- Créer une variable cible artificielle `is_hoax`.
- Compter les relevés étiquetés comme canulars.
- Examiner des exemples.
- Identifier les limites de la règle.

## Imports

In [1]:
from pathlib import Path
import csv
import re
import pandas as pd

## Chemins et colonnes

In [2]:
DATA_PATH = Path("../data/releves_klaxo3.csv")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]

## Recharger les lignes valides

In [3]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)

    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append({
                "numero_ligne": numero_ligne,
                "nb_champs": len(row),
                "contenu": row
            })

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

print(f"Nombre de lignes exploitables : {len(df)}")
print(f"Nombre de lignes structurées à part : {len(lignes_problemes)}")

Nombre de lignes exploitables : 88679
Nombre de lignes structurées à part : 196


## Refaire les conversions utiles

In [4]:
colonnes_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
]

colonnes_dates = [
    "datetime",
    "date_posted",
]

for col in colonnes_numeriques:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in colonnes_dates:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration_seconds             float64
duration_hours_min            object
comments                      object
date_posted           datetime64[ns]
latitude                     float64
longitude                    float64
dtype: object

## Afficher quelques commentaires

In [7]:
pd.set_option("display.max_colwidth", None)

df[
    [
        "datetime",
        "city",
        "country",
        "shape",
        "comments"
    ]
].sample(
    n=10,
    random_state=42
)

,datetime,city,country,shape,comments
48205,2004-05-21 22:45:00,st-philippe (canada),,triangle,HBCCUFO CANADIAN REPORT: triangle
73977,1975-08-16 20:00:00,highlands,us,fireball,Ball of light like a roman candle
66019,2013-07-24 00:00:00,tulsa,us,light,Green beam of light over Tulsa
38590,2009-03-05 20:05:00,west valley city,,circle,Objects hovering near and above Salt Lake City and other one flying in zig zag pattern
29694,2013-02-15 19:00:00,arlington heights,us,fireball,We saw 4 big bright orange fire ball type flying slowly and disappearing 1 by 1.
5719,2012-10-28 16:00:00,bourbonnais,us,circle,Metallic blue round object over bourbonnais&#44 IL dashes out of sight in 3 seconds
12066,1998-01-01 20:00:00,graniteville,us,disk,I was taking pictures of a full moon and did not see it until the pictures were developed. I don&#39t know what to do with the picture (it
618,1985-10-01 05:30:00,monroe,us,sphere,Years ago&#44 my husband&#39s aunt and I witnessed something very strange on the way to work early one morning. It was about 5:30am and we we
78087,2005-08-30 21:30:00,kentfield,us,disk,UFO exiting our atmosphere.
61290,1980-07-01 22:00:00,netherlands,,unknown,Big black object with coloured lights


## Afficher les mots liés à des canulars

In [8]:
df.loc[
    df["comments"].fillna("").str.contains(
        "hoax|fake|prank|joke",
        case=False,
        regex=True
    ),
    [
        "datetime",
        "city",
        "country",
        "comments"
    ]
].head(20)

,datetime,city,country,comments
658,1994-10-01 13:13:00,new york city,us,a flying colorful disc above my car&#44 near Erie. ((NUFORC Note: Possible hoax?? PD))
808,2004-10-01 17:00:00,las vegas,us,((HOAX??)) Short encounter with space craft on my way into my parking lot area.
958,2008-10-01 19:12:00,bonham,us,Silver egg shape over six houses. ((NUFORC Note: Possible hoax?? PD))
1207,2007-10-12 22:00:00,irvine,us,Lights in Irvine October 2007: Hoax
1211,2007-10-12 23:00:00,rogers,us,((HOAX??)) abduction. 500 Lights On Object0: Yes
1492,2009-10-13 06:30:00,troy,us,((HOAX??)) Flying craft which was big as a football field
1582,2013-10-13 10:50:00,santa fe,us,((HOAX??)) Some kind of aircraft with a HUGE wingspan was flying very low over my neighborhood in Santa Fe&#44 NM.
1727,2006-10-14 02:00:00,yuma,us,((HOAX??)) two aliens appeared from a bright light to peacefully investigate the surroundings in the woods
1740,2007-10-14 05:00:00,rawalpindi (pakistan),,((HOAX)) usaually i stand near the airport. on that day i saw that there were
1965,NaT,greenwich,us,((HOAX??)) had no control on tv. ash tray killd itself. the obkect was one big light.


## Définir la règle

### Règle retenue

Un relevé est étiqueté comme canular lorsque son commentaire contient au moins
un mot-clé explicitement associé à une fraude, une mise en scène ou une
plaisanterie.

In [9]:
MOTS_CLES_CANULAR = [
    "hoax",
    "fake",
    "prank",
    "joke",
    "not real",
    "made up",
    "fraud",
]

pattern_canular = "|".join(
    re.escape(mot) for mot in MOTS_CLES_CANULAR
)

print("Expression recherchée :")
print(pattern_canular)

Expression recherchée :
hoax|fake|prank|joke|not\ real|made\ up|fraud


## Créer la cible

In [10]:
df["comments_clean"] = (
    df["comments"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["is_hoax"] = (
    df["comments_clean"]
    .str.contains(
        pattern_canular,
        regex=True,
        na=False
    )
    .astype(int)
)

df[
    [
        "comments",
        "is_hoax"
    ]
].head(10)

,comments,is_hoax
0,This event took place in early fall around 1949-50. It occurred after a Boy Scout meeting in the Baptist Church. The Baptist Church sit,0
1,1949 Lackland AFB&#44 TX. Lights racing across the sky &amp; making 90 degree turns on a dime.,0
2,Green/Orange circular disc over Chester&#44 England,0
3,My older brother and twin sister were leaving the only Edna theater at about 9 PM&#44...we had our bikes and I took a different route home,0
4,AS a Marine 1st Lt. flying an FJ4B fighter/attack aircraft on a solo night exercise&#44 I was at 50&#44000&#39 in a &quot;clean&quot; aircraft (no ordinan,0
5,My father is now 89 my brother 52 the girl with us now 51 myself 49 and the other fellow which worked with my father if he&#39s still livi,0
6,penarth uk circle 3mins stayed 30ft above me for 3 mins slowly moved of and then with the blink of the eye the speed was unreal,0
7,A bright orange color changing to reddish color disk/saucer was observed hovering above power transmission lines.,0
8,Strobe Lighted disk shape object observed close&#44 at low speeds&#44 and low altitude in Oct 1966 in Pell City Alabama,0
9,Saucer zaps energy from powerline as my pregnant mother receives mental signals not to pass info,0


## Compter les relevés marqués

In [11]:
nombre_total = len(df)
nombre_canulars = int(df["is_hoax"].sum())
proportion_canulars = df["is_hoax"].mean()

print(f"Nombre total de relevés : {nombre_total}")
print(f"Nombre de relevés étiquetés canulars : {nombre_canulars}")
print(f"Proportion de canulars : {proportion_canulars:.2%}")

Nombre total de relevés : 88679
Nombre de relevés étiquetés canulars : 869
Proportion de canulars : 0.98%


## Répartition des deux classes 

In [12]:
print("Nombre de lignes par classe :")
display(df["is_hoax"].value_counts())

print("\nProportion par classe :")
display(
    df["is_hoax"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Nombre de lignes par classe :


is_hoax
0    87810
1      869
Name: count, dtype: int64


Proportion par classe :


is_hoax
0    99.02
1     0.98
Name: proportion, dtype: float64

## Examiner les exemples détectés

In [13]:
exemples_canulars = df.loc[
    df["is_hoax"] == 1,
    [
        "datetime",
        "city",
        "state",
        "country",
        "shape",
        "duration_seconds",
        "comments"
    ]
]

exemples_canulars.sample(
    n=min(20, len(exemples_canulars)),
    random_state=42
)

,datetime,city,state,country,shape,duration_seconds,comments
38464,NaT,venda (south africa),,,light,60.0,((HOAX??)) Neet 2 tolk 2 some 1.
8344,NaT,islamabad (pakistan),,,circle,10.0,((HOAX)) saw a ufo that flyed over the house and a hill
48281,2011-05-21 20:00:00,toronto (canada),on,ca,triangle,60.0,((HOAX)) triangle 8 yellow lights over Toronto&#44 Ontario&#44 Canada.
8354,2010-01-10 20:00:00,butler,mo,us,chevron,120.0,((HOAX??)) I was heading driving to the store when I seen a red and yellow lighted craft in the air flying 1000 ft or less strobing.
87309,1999-09-04 22:30:00,summerville,sc,us,circle,900.0,((HOAX??)) I was stopped on dirt road in the woods and two bright objects passed over me.
73789,2007-08-15 02:00:00,jacksonville,fl,us,fireball,0.0,((HOAX??)) Fire ball with red lights&#44 red tail behind it and a tall object. ((NUFORC Note: Student report. PD))
10609,NaT,farmington,mo,us,,0.0,I am the person who submitted the ufo report from Farmington&#44 Mo. I realize my report may be deemed a hoax&#44because I gave no contact in
79117,2004-08-05 22:00:00,prescott,fl,,triangle,240.0,((NUFORC Note: Possible hoax. No Prescott in FL; Yavapai County is in AZ. PD)) Triangular craft with bright lights on the points&#8230;
58280,2013-06-29 22:30:00,indianapolis,in,us,cylinder,15.0,((HOAX??)) Massive cylinder shaped object&#44 thousands of feet long. ((NUFORC Note: Witness provides no personal data. PD))
45264,2011-04-08 23:00:00,irmo,sc,us,triangle,0.0,((HOAX??)) ufo


## Examiner le ou les mots ayant détecter le label

In [14]:
def mots_declencheurs(commentaire):
    texte = str(commentaire).lower()

    return [
        mot
        for mot in MOTS_CLES_CANULAR
        if mot in texte
    ]

df["mots_declencheurs"] = df["comments"].apply(
    mots_declencheurs
)

df.loc[
    df["is_hoax"] == 1,
    [
        "comments",
        "mots_declencheurs",
        "is_hoax"
    ]
].head(30)

,comments,mots_declencheurs,is_hoax
658,a flying colorful disc above my car&#44 near Erie. ((NUFORC Note: Possible hoax?? PD)),[hoax],1
670,The object was made up of a formation of small lights&#44 plus one light moving independantly around main formation,[made up],1
808,((HOAX??)) Short encounter with space craft on my way into my parking lot area.,[hoax],1
958,Silver egg shape over six houses. ((NUFORC Note: Possible hoax?? PD)),[hoax],1
1207,Lights in Irvine October 2007: Hoax,[hoax],1
1211,((HOAX??)) abduction. 500 Lights On Object0: Yes,[hoax],1
1492,((HOAX??)) Flying craft which was big as a football field,[hoax],1
1582,((HOAX??)) Some kind of aircraft with a HUGE wingspan was flying very low over my neighborhood in Santa Fe&#44 NM.,[hoax],1
1727,((HOAX??)) two aliens appeared from a bright light to peacefully investigate the surroundings in the woods,[hoax],1
1740,((HOAX)) usaually i stand near the airport. on that day i saw that there were,[hoax],1


## Compter les mots-clés qui apparaissent

In [15]:
compte_declencheurs = {
    mot: int(
        df["comments_clean"].str.contains(
            re.escape(mot),
            regex=True,
            na=False
        ).sum()
    )
    for mot in MOTS_CLES_CANULAR
}

df_compte_declencheurs = (
    pd.DataFrame.from_dict(
        compte_declencheurs,
        orient="index",
        columns=["nombre_de_commentaires"]
    )
    .rename_axis("mot_cle")
    .sort_values(
        "nombre_de_commentaires",
        ascending=False
    )
)

df_compte_declencheurs

,nombre_de_commentaires
mot_cle,
hoax,802
made up,26
joke,17
not real,13
fake,9
prank,3
fraud,0


## Exporter les résultats

In [16]:
df.loc[
    df["is_hoax"] == 1,
    [
        "datetime",
        "city",
        "state",
        "country",
        "shape",
        "duration_seconds",
        "comments",
        "mots_declencheurs",
        "is_hoax"
    ]
].to_csv(
    OUTPUT_DIR / "releves_etiquetes_canulars.csv",
    index=False
)

df_compte_declencheurs.to_csv(
    OUTPUT_DIR / "compte_mots_cles_canular.csv",
    index=True
)

print("Fichiers créés :")
print(OUTPUT_DIR / "releves_etiquetes_canulars.csv")
print(OUTPUT_DIR / "compte_mots_cles_canular.csv")

Fichiers créés :
..\outputs\releves_etiquetes_canulars.csv
..\outputs\compte_mots_cles_canular.csv
